# EDGAR XBRL JSON – Datenstruktur & Normalizer-Funktionen

Dieses Notebook erklärt:
1. Wie die EDGAR-JSON-API aufgebaut ist
2. Was `resolve_tag`, `is_duration`, `classify_period`, `ytd_to_quarterly` machen
3. Warum diese Funktionen überhaupt nötig sind

Wir arbeiten mit echten Apple-Daten von EDGAR.

---
## 0. Setup

In [ ]:
import requests, json
from datetime import date
from pprint import pprint

HEADERS = {"User-Agent": "edgar-notebook david.steimel02@gmail.com"}

APPLE_CIK = "0000320193"
FACTS_URL  = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{APPLE_CIK}.json"

facts = requests.get(FACTS_URL, headers=HEADERS).json()
print("Top-Level-Keys:", list(facts.keys()))

: 

---
## 1. Gesamtstruktur der EDGAR CompanyFacts-JSON

Die Antwort hat drei Ebenen:

```
facts_json
├── cik          (int)
├── entityName   (str)
└── facts
    ├── us-gaap          ← US-GAAP XBRL Tags
    │   ├── NetIncomeLoss
    │   │   ├── label
    │   │   ├── description
    │   │   └── units
    │   │       └── USD         ← Einheit
    │   │           └── [ fact1, fact2, ... ]   ← Liste von Einzelmeldungen
    │   ├── Assets
    │   └── ...
    └── dei              ← Document & Entity Information (z.B. SharesOutstanding)
```

**Kernpunkt:** Jeder XBRL-Tag enthält eine Liste von Fact-Dictionaries. 
Jedes Dict ist eine einzelne Zeile aus einem 10-K oder 10-Q.

In [ ]:
# Welche Namespaces gibt es?
print("Namespaces in facts:", list(facts["facts"].keys()))

# Wie viele us-gaap Tags hat Apple?
us_gaap = facts["facts"]["us-gaap"]
print(f"Anzahl us-gaap Tags bei Apple: {len(us_gaap)}")

In [3]:
# Zoom in: ein konkreter Tag – NetIncomeLoss
ni_tag = us_gaap["NetIncomeLoss"]

print("Keys eines Tags:", list(ni_tag.keys()))
print("Label:", ni_tag["label"])
print("Einheiten:", list(ni_tag["units"].keys()))

NameError: name 'us_gaap' is not defined

In [ ]:
# Die eigentlichen Fact-Einträge (in USD)
ni_facts = ni_tag["units"]["USD"]

print(f"Anzahl Fact-Einträge für NetIncomeLoss: {len(ni_facts)}")
print("\nDie ersten 5 Einträge:")
pprint(ni_facts[:5])

---
## 2. Anatomie eines einzelnen Fact-Dicts

Jedes Dict in dieser Liste entspricht einer Zeile aus einem SEC-Filing.

**Zwei Typen:**

### Typ A: *Instant* (Stichtagswert)
```python
{
  'end':    '2023-09-30',   # Bilanzstichtag
  'val':    352583000000,   # Wert
  'accn':   '0000320193-23-000106',
  'fy':     2023,
  'fp':     'FY',
  'form':   '10-K',
  'filed':  '2023-11-03',
  'frame':  'CY2023Q3I'
}
```
→ **Kein `start`-Key!** Das ist z.B. Assets oder Liabilities: Wert gilt *am* Stichtag.

### Typ B: *Duration* (Periodenwert)
```python
{
  'start':  '2022-10-01',   # Beginn der Periode
  'end':    '2023-09-30',   # Ende der Periode  
  'val':    96995000000,
  'accn':   '0000320193-23-000106',
  'fy':     2023,
  'fp':     'FY',
  'form':   '10-K',
  'filed':  '2023-11-03',
  'frame':  'CY2023'
}
```
→ **Hat `start` UND `end`!** Das sind GuV/CFS-Werte: Wert *über* eine Periode.

Das ist der Unterschied den `is_duration()` prüft.

In [ ]:
# Zeige beide Typen im Vergleich
instant_examples  = [f for f in ni_facts if "start" not in f][:2]
duration_examples = [f for f in ni_facts if "start" in f][:2]

print("=== INSTANT (kein 'start') ===")
pprint(instant_examples)

print("\n=== DURATION (hat 'start') ===")
pprint(duration_examples)

In [ ]:
# is_duration() macht genau das:
def is_duration(fact: dict) -> bool:
    return "start" in fact

# Teste:
print("Instant fact ist duration:", is_duration(instant_examples[0]))   # False
print("Duration fact ist duration:", is_duration(duration_examples[0])) # True

---
## 3. Das YTD-Problem – warum `classify_period` nötig ist

Das ist der entscheidende Punkt. Stell dir vor, Apple meldet im 10-Q für Q3:

```
NetIncomeLoss für Periode 2023-01-01 bis 2023-09-30 = 73 Mrd. $
```

Das ist **kein Q3-Wert** – das ist der **Year-to-Date (YTD) Wert** für 9 Monate!

EDGAR liefert grundsätzlich das was im Filing steht. Bei der GuV und dem CFS 
steht im 10-Q immer der **kumulierte Wert seit Jahresbeginn**, nicht der Quartalswert.

**Compustat macht daraus echte Quartalswerte durch Differenzbildung:**
```
Q3 = 9M-YTD  - H1-YTD
Q2 = H1-YTD  - Q1
Q4 = FY      - 9M-YTD
Q1 = Q1-YTD  (kein Problem, ist direkt der Quartalswert)
```

`classify_period()` schaut auf die Länge der Periode (end - start in Tagen)
und entscheidet: ist das ein Q1-YTD (~90 Tage), H1-YTD (~180 Tage), 
9M-YTD (~270 Tage), oder FY (~365 Tage)?

In [ ]:
# Zeige alle Duration-Facts für NetIncomeLoss mit ihrer Perioden-Länge
dur_facts = [f for f in ni_facts if is_duration(f)]

print(f"{'start':12} {'end':12} {'days':>5}  {'form':5}  {'fp':4}  {'val':>20}")
print("-" * 70)
for f in sorted(dur_facts, key=lambda x: x["end"])[-20:]:  # letzte 20
    start_d = date.fromisoformat(f["start"])
    end_d   = date.fromisoformat(f["end"])
    days    = (end_d - start_d).days
    print(f"{f['start']:12} {f['end']:12} {days:>5}  {f['form']:5}  {f['fp']:4}  {f['val']:>20,}")

Du siehst jetzt das Muster:
- ~90 Tage = Q1 YTD (aus 10-Q)
- ~181 Tage = H1 YTD (aus 10-Q)
- ~273 Tage = 9M YTD (aus 10-Q)
- ~365 Tage = Full Year (aus 10-K)

Dieselbe Zahl taucht mehrfach auf – EDGAR dupliziert Facts wenn derselbe Wert
in mehreren Filings auftaucht (z.B. Vorjahreswerte die im aktuellen 10-Q mitgeliefert werden).

In [ ]:
def classify_period(start_str: str, end_str: str) -> str:
    start_d = date.fromisoformat(start_str)
    end_d   = date.fromisoformat(end_str)
    days    = (end_d - start_d).days
    if  75 <= days <= 105:  return "Q1"   # ~90 Tage
    if 165 <= days <= 195:  return "H1"   # ~180 Tage
    if 255 <= days <= 285:  return "9M"   # ~270 Tage
    if 350 <= days <= 380:  return "A"    # ~365 Tage
    return "SKIP"

# Teste mit echten Werten:
for f in sorted(dur_facts, key=lambda x: x["end"])[-8:]:
    pt = classify_period(f["start"], f["end"])
    days = (date.fromisoformat(f["end"]) - date.fromisoformat(f["start"])).days
    print(f"{f['start']} → {f['end']}  ({days:3d} Tage)  → {pt}")

---
## 4. `resolve_tag` – warum es eine Priority-Liste braucht

Verschiedene Firmen nutzen unterschiedliche XBRL-Tags für dasselbe Konzept.
Apple nutzt vielleicht `CashAndCashEquivalentsAtCarryingValue`,
Microsoft `CashCashEquivalentsAndShortTermInvestments`.

Deine `config.py` definiert pro Compustat-Variable eine **geordnete Fallback-Liste**.
`resolve_tag()` nimmt die JSON und diese Liste, und gibt die Facts des 
**ersten Tags zurück das tatsächlich in den Daten existiert**.

In [ ]:
def resolve_tag(facts_json: dict, tag_priorities: list) -> list:
    us_gaap_data = facts_json.get("facts", {}).get("us-gaap", {})
    for sec_tag in tag_priorities:
        if sec_tag in us_gaap_data:
            units_dict = us_gaap_data[sec_tag].get("units", {})
            for unit_key, facts_list in units_dict.items():
                print(f"  → Gefunden: Tag='{sec_tag}', Unit='{unit_key}', {len(facts_list)} Einträge")
                return facts_list
    return []

# Beispiel: che (Cash & Equivalents) mit Priority-Liste
che_priorities = [
    "CashAndCashEquivalentsAtCarryingValue",          # spezifischer
    "CashCashEquivalentsAndShortTermInvestments",     # breiter, Fallback
]

print("Suche che für Apple:")
che_facts = resolve_tag(facts, che_priorities)
print(f"\nErgebnis: {len(che_facts)} Facts")
print("Letzter Eintrag:")
pprint(sorted(che_facts, key=lambda x: x["end"])[-1])

In [ ]:
# Was passiert wenn der erste Tag fehlt? Simuliere mit falschem primären Tag:
print("Suche mit nicht-existierentem primären Tag:")
test = resolve_tag(facts, ["NichtExistierenderTag", "CashAndCashEquivalentsAtCarryingValue"])
print(f"Ergebnis: {len(test)} Facts")

---
## 5. `ytd_to_quarterly` – der Kern des Normalizers

Jetzt kommt alles zusammen. Diese Funktion nimmt die rohen EDGAR-Facts
(die YTD-Werte für ein Duration-Tag wie NetIncomeLoss) und rechnet sie
in **echte Quartalswerte** um.

**Algorithmus:**

1. Gruppiere alle Facts nach `start`-Datum → das identifiziert das Fiskaljahr
   (alle Facts innerhalb eines FY haben denselben FY-Beginn)

2. Für jede FY-Gruppe: baue ein Lookup `{period_type → (value, end_date, form)}`
   via `classify_period`

3. Berechne echte Quartalswerte durch Differenzbildung:
   - Q1 = Q1-YTD (direkt)
   - Q2 = H1-YTD − Q1-YTD  
   - Q3 = 9M-YTD − H1-YTD
   - Q4 = FY − 9M-YTD

4. Wenn ein YTD-Wert fehlt → überspringe das Quartal

**Achtung: Das ist die Logik die du noch implementieren musst (das `...` im Code)!**

In [ ]:
# Schritt 1: Zeige die Gruppenstruktur für NetIncomeLoss
dur_ni = [f for f in ni_facts if is_duration(f)]

# Gruppiere nach FY-Start
by_fy = {}
for rec in dur_ni:
    fy_start = rec["start"]
    by_fy.setdefault(fy_start, []).append(rec)

# Zeige die letzten 2 Fiskaljahre
recent_fy_starts = sorted(by_fy.keys())[-2:]

for fy_start in recent_fy_starts:
    print(f"\n=== FY Start: {fy_start} ===")
    for rec in sorted(by_fy[fy_start], key=lambda x: x["end"]):
        pt = classify_period(rec["start"], rec["end"])
        days = (date.fromisoformat(rec["end"]) - date.fromisoformat(rec["start"])).days
        print(f"  {rec['start']} → {rec['end']}  ({days:3d}d)  {pt:4}  form={rec['form']:5}  val={rec['val']:>20,}")

In [ ]:
# Schritt 2: Lookup-Dict bauen für ein konkretes FY
target_fy = sorted(by_fy.keys())[-1]  # letztes verfügbares FY
fy_records = by_fy[target_fy]

lookup = {}
for rec in fy_records:
    ptype = classify_period(rec["start"], rec["end"])
    if ptype != "SKIP":
        # Bei Duplikaten: neuestes Filing gewinnt (nach 'filed' sortieren)
        if ptype not in lookup or rec["filed"] > lookup[ptype][2]:
            lookup[ptype] = (float(rec["val"]), rec["end"], rec["form"])

print(f"Lookup für FY ab {target_fy}:")
for ptype, (val, end, form) in sorted(lookup.items()):
    print(f"  {ptype:3}  end={end}  form={form:5}  val={val:>20,.0f}")

In [ ]:
# Schritt 3: Differenzbildung – so sieht die korrekte Logik aus
# (Das ist die Implementierung die du in ytd_to_quarterly einbauen musst)

q1_data = lookup.get("Q1")
h1_data = lookup.get("H1")
nm_data = lookup.get("9M")
a_data  = lookup.get("A")

quarterly_results = []

# Q1: direkt verfügbar (YTD von 0 bis Q1 Ende = Q1)
if q1_data:
    val, end, form = q1_data
    quarterly_results.append(("Q1", end, val, form))

# Q2: H1-YTD minus Q1-YTD
if h1_data and q1_data:
    val = h1_data[0] - q1_data[0]
    quarterly_results.append(("Q2", h1_data[1], val, h1_data[2]))

# Q3: 9M-YTD minus H1-YTD
if nm_data and h1_data:
    val = nm_data[0] - h1_data[0]
    quarterly_results.append(("Q3", nm_data[1], val, nm_data[2]))

# Q4: FY minus 9M-YTD
if a_data and nm_data:
    val = a_data[0] - nm_data[0]
    quarterly_results.append(("Q4", a_data[1], val, a_data[2]))

print(f"Echte Quartalswerte für NetIncomeLoss (FY ab {target_fy}):")
print(f"  (Mio. USD)")
for qname, end, val, form in quarterly_results:
    print(f"  {qname}  period_end={end}  form={form:5}  val={val/1e6:>10,.1f} Mio.")

# Verifikation: Summe aller Quartale sollte = FY-Wert sein
if a_data and len(quarterly_results) == 4:
    q_sum = sum(v for _, _, v, _ in quarterly_results)
    fy_val = a_data[0]
    diff = abs(q_sum - fy_val)
    print(f"\nVerifikation: Summe Q1-Q4 = {q_sum/1e6:,.1f}  |  FY = {fy_val/1e6:,.1f}  |  Differenz = {diff:.0f}")

---
## 6. Was noch fehlt: Duplikate

Wenn du oben die rohen Facts angeschaut hast, siehst du dasselbe `(start, end)`-Paar
manchmal mehrfach – mit verschiedenen `accn`-Nummern (Accession Numbers).

Das passiert weil:
- Ein 10-K enthält Vorjahreswerte als Vergleich → taucht zweimal auf
- Amended Filings (10-K/A) überschreiben ältere Werte

Strategie: **neuestes `filed`-Datum gewinnt** (wie im lookup-Schritt oben bereits gemacht).

In [ ]:
# Zeige Duplikate für ein FY
from collections import Counter

period_keys = [(f["start"], f["end"]) for f in fy_records]
dupes = {k: v for k, v in Counter(period_keys).items() if v > 1}

if dupes:
    print("Doppelte (start, end)-Paare in diesem FY:")
    for (s, e), count in dupes.items():
        print(f"  {s} → {e}  ({count}x)")
        for rec in [r for r in fy_records if r["start"] == s and r["end"] == e]:
            print(f"    filed={rec['filed']}  accn={rec['accn']}  val={rec['val']:,}")
else:
    print("Keine Duplikate in diesem FY (probiere ein älteres FY)")

# Schaue in allen FYs nach Duplikaten
all_dur = [f for f in ni_facts if is_duration(f)]
all_keys = [(f["start"], f["end"]) for f in all_dur]
all_dupes = {k: v for k, v in Counter(all_keys).items() if v > 1}
print(f"\nGesamt: {len(all_dupes)} doppelte Perioden über alle FYs")

---
## 7. Instant-Tags brauchen kein YTD-to-Quarterly

Balance Sheet Werte (Assets, Liabilities, Equity etc.) sind **Instant**-Werte.
Sie gelten zu einem bestimmten Stichtag, nicht über eine Periode.
Dafür gibt es kein YTD-Problem – du nimmst einfach den Wert am Quartalsende.

Der Collector filtert diese dann nach `form = '10-Q'` (Quartale) oder `'10-K'` (Jahresende).

In [ ]:
# Assets (at) – Instant-Tag
at_facts = us_gaap["Assets"]["units"]["USD"]

print("Letzte 6 Assets-Werte (Instant, kein YTD-Problem):")
print(f"  {'end':12}  {'form':5}  {'val':>20}")
print("  " + "-"*45)
for f in sorted(at_facts, key=lambda x: x["end"])[-6:]:
    # Kein 'start' → kein classify_period nötig
    has_start = "start" in f
    print(f"  {f['end']:12}  {f['form']:5}  {f['val']:>20,}  (has_start={has_start})")

---
## 8. Zusammenfassung: Welche Funktion macht was?

| Funktion | Input | Output | Zweck |
|---|---|---|---|
| `is_duration(fact)` | ein Fact-Dict | `True/False` | Unterscheidet Instant (Bilanz) von Duration (GuV/CFS) |
| `classify_period(start, end)` | zwei ISO-Datum-Strings | `"Q1"`, `"H1"`, `"9M"`, `"A"`, `"SKIP"` | Bestimmt ob ein Duration-Fact ein Quartal, Halbjahr, 9M oder FY abdeckt |
| `resolve_tag(facts_json, priorities)` | gesamte CompanyFacts-JSON + Tag-Liste | Liste von Fact-Dicts | Findet das beste verfügbare XBRL-Tag anhand der Priority-Liste |
| `ytd_to_quarterly(cik, tag, records)` | rohe EDGAR Duration-Facts | Liste von `FactRecord`-Objekten | Rechnet YTD-Werte in echte Quartalswerte um (Differenzbildung) |

**Das fehlende TODO in deinem Code** ist Schritt 2c-f in `ytd_to_quarterly`:
die Differenzbildung Q1, Q2=H1−Q1, Q3=9M−H1, Q4=FY−9M,
und daraus `FactRecord`-Objekte bauen.

In [ ]:
# Vollständige ytd_to_quarterly Implementierung (mit dem gefüllten TODO)
from dataclasses import dataclass
from datetime import date

@dataclass
class FactRecord:
    cik: str
    tag: str
    period_end: date
    period_type: str   # "Q" oder "A"
    value: float
    source_form: str

def ytd_to_quarterly(cik: str, compustat_tag: str, records: list) -> list:
    # Schritt 1: Gruppiere nach FY-Start
    by_fy = {}
    for rec in records:
        if not is_duration(rec):
            continue
        by_fy.setdefault(rec["start"], []).append(rec)

    result = []

    for fy_start, fy_records in by_fy.items():
        # Schritt 2a: Lookup bauen; bei Duplikaten: neuestes 'filed' gewinnt
        lookup = {}
        for rec in fy_records:
            ptype = classify_period(rec["start"], rec["end"])
            if ptype == "SKIP":
                continue
            existing = lookup.get(ptype)
            if existing is None or rec["filed"] > existing[2]:
                lookup[ptype] = (float(rec["val"]), rec["end"], rec["form"])

        # Schritt 2b: YTD-Werte extrahieren
        q1_data = lookup.get("Q1")
        h1_data = lookup.get("H1")
        nm_data = lookup.get("9M")
        a_data  = lookup.get("A")

        # Schritt 2c–f: Differenzbildung → echte Quartalswerte
        if q1_data:
            result.append(FactRecord(
                cik=cik, tag=compustat_tag,
                period_end=date.fromisoformat(q1_data[1]),
                period_type="Q", value=q1_data[0], source_form=q1_data[2]
            ))

        if h1_data and q1_data:
            result.append(FactRecord(
                cik=cik, tag=compustat_tag,
                period_end=date.fromisoformat(h1_data[1]),
                period_type="Q", value=h1_data[0] - q1_data[0], source_form=h1_data[2]
            ))

        if nm_data and h1_data:
            result.append(FactRecord(
                cik=cik, tag=compustat_tag,
                period_end=date.fromisoformat(nm_data[1]),
                period_type="Q", value=nm_data[0] - h1_data[0], source_form=nm_data[2]
            ))

        if a_data and nm_data:
            result.append(FactRecord(
                cik=cik, tag=compustat_tag,
                period_end=date.fromisoformat(a_data[1]),
                period_type="Q", value=a_data[0] - nm_data[0], source_form=a_data[2]
            ))

    return sorted(result, key=lambda r: r.period_end)


# Test mit echten Apple-Daten
ni_records = ytd_to_quarterly(APPLE_CIK, "ni", ni_facts)
print(f"Erzeugte {len(ni_records)} quarterly FactRecords für Apple NetIncomeLoss")
print("\nLetzte 8:")
for r in ni_records[-8:]:
    print(f"  {r.period_end}  {r.period_type}  form={r.source_form:5}  val={r.value/1e6:>10,.1f} Mio.")

---
## 9. Was bleibt offen?

Ein Randfall den du noch bedenken musst: **nicht alle Firmen haben ein Fiskaljahr das am 1.1. beginnt.**
Apple's FY beginnt z.B. im Oktober. Das spielt für die Differenzbildung keine Rolle
(wir gruppieren ja nach `start`-Datum), aber es bedeutet dass Q1 für Apple 
Oktober–Dezember ist, nicht Januar–März.

Das `period_end`-Datum im `FactRecord` ist daher wichtiger als ein abstraktes "Q1/Q2/Q3/Q4" Label –
für deine Signal-Berechnung willst du sowieso auf dem `period_end` joinen,
nicht auf Quartalsnummern.